In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

###############################################################
# Customer Segmentation with RFM
###############################################################

# Customer Segmentation with RFM in 6 Steps

# 1. Business Problem
# 2. Data Understanding
# 3. Data Preparation
# 4. Calculating RFM Metrics
# 5. Calculating RFM Scores
# 6. Naming & Analysing RFM Segments

# An e-commerce company wants to segment its customers and determine marketing strategies according to these segments.
# In these behaviors we will create linked groups based on chunks. 

Dataset story
1.Invoice no:Unique invoice number if number starting with 'C' its been cancelled
2.StockCode:Product number.Unique number for every product
3.Description:Product name
4.Quantity:sold product quantity
5.InvoiceDate:Date and time of Invoice
6.UnitPrice:Product price
7.CustomerID:Unique customer number
8.Country:Country name



In [ ]:
###############################################################
# Data Understanding
###############################################################

In [ ]:
import datetime as dt
pd.set_option('display.max_columns', None)
!pip install xlrd
!pip install openpyxl

In [ ]:
df_ = pd.read_excel("../input/online-retail-ii-data-set-from-ml-repository/online_retail_II.xlsx",
                    sheet_name="Year 2009-2010")

In [ ]:
df = df_.copy()

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

**How many unique product ?**

In [ ]:
df.Description.nunique()

**How many of which products are there? **

In [ ]:
df["Description"].value_counts().head()

*Which one is the most ordered product,and how to sort*

In [ ]:
df.groupby("Description").agg({"Quantity": "sum"}).head()

In [ ]:
df.groupby("Description").agg({"Quantity": "sum"}).sort_values("Quantity", ascending=False).head()

**How many invoices have been issued in total**

In [ ]:
df["Invoice"].nunique()

How much does company earned in total?

In [ ]:
df = df[~df["Invoice"].str.contains("C", na=False)]
df["TotalPrice"] = df["Quantity"] * df["Price"]

which are the most expensive products

In [ ]:
df.sort_values("Price", ascending=False).head()

how many order has been issued per country in total

In [ ]:
df["Country"].value_counts()

how much does the company earned by countries

In [ ]:
df.groupby("Country").agg({"TotalPrice": "sum"}).sort_values("TotalPrice", ascending=False).head()

###############################################################
# Data Preparation
###############################################################

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.describe([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T

###############################################################
# Calculating RFM Metrics
###############################################################
# Recency, Frequency, Monetary

In [ ]:
df["InvoiceDate"].max()

In [ ]:
today_date = dt.datetime(2010, 12, 11)

In [ ]:
rfm = df.groupby('Customer ID').agg({'InvoiceDate': lambda date: (today_date - date.max()).days,
                                     'Invoice': lambda num: len(num),
                                     'TotalPrice': lambda TotalPrice: TotalPrice.sum()})

In [ ]:
rfm.columns = ['Recency', 'Frequency', 'Monetary']

In [ ]:
rfm = rfm[(rfm["Monetary"]) > 0 & (rfm["Frequency"] > 0)]


###############################################################
# Calculating RFM Scores
###############################################################

In [ ]:
rfm["RecencyScore"] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])

In [ ]:
rfm["FrequencyScore"] = pd.qcut(rfm['Frequency'], 5, labels=[1, 2, 3, 4, 5])

In [ ]:
rfm["MonetaryScore"] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

In [ ]:
rfm["RFM_SCORE"] = (rfm['RecencyScore'].astype(str) +
                    rfm['FrequencyScore'].astype(str) +
                    rfm['MonetaryScore'].astype(str))

In [ ]:
rfm[rfm["RFM_SCORE"] == "555"].head()

In [ ]:
rfm[rfm["RFM_SCORE"] == "111"]

###############################################################
# Naming & Analysing RFM Segments
###############################################################

In [ ]:
seg_map = {
    r'[1-2][1-2]': 'Hibernating',
    r'[1-2][3-4]': 'At_Risk',
    r'[1-2]5': 'Cant_Loose',
    r'3[1-2]': 'About_to_Sleep',
    r'33': 'Need_Attention',
    r'[3-4][4-5]': 'Loyal_Customers',
    r'41': 'Promising',
    r'51': 'New_Customers',
    r'[4-5][2-3]': 'Potential_Loyalists',
    r'5[4-5]': 'Champions'
}

In [ ]:
rfm['Segment'] = rfm['RecencyScore'].astype(str) + rfm['FrequencyScore'].astype(str)

In [ ]:
rfm['Segment'] = rfm['Segment'].replace(seg_map, regex=True)

In [ ]:
df[["Customer ID"]].nunique()

In [ ]:
rfm[["Segment", "Recency", "Frequency", "Monetary"]].groupby("Segment").agg(["mean", "count"])

In [ ]:
rfm[rfm["Segment"] == "Need_Attention"].head()

In [ ]:
rfm[rfm["Segment"] == "Need_Attention"].index

In [ ]:
new_df = pd.DataFrame()

In [ ]:
new_df["Need_Attention"] = rfm[rfm["Segment"] == "Need_Attention"].index

In [ ]:
new_df.to_csv("Need_Attention.csv")